# CSV 文件的 Simple RAG：把“表格行”变成可检索知识

这一节实现一个最小可用的 **CSV RAG**：

- 把 CSV 的每一行（客户信息）变成一个 `Document`
- 用 embedding 建一个向量库（FAISS）
- 用 retriever 检索最相关的行
- 把检索到的行拼进 prompt，让 LLM 回答

适用场景：你有大量结构化表格数据，但希望用“自然语言问答”的方式查询（先不做 SQL/筛选器，只做语义检索 + 生成）。

## 0) 环境准备

本 notebook 不包含 `pip install`。你需要确保已安装：

- `langchain` / `langchain-core`
- `langchain-openai`
- `langchain-community`（含 `CSVLoader`、`FAISS`）
- `faiss-cpu`
- `pandas`
- `python-dotenv`

并已设置 `OPENAI_API_KEY`（例如放在 `.env`）。

In [1]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

DATA_DIR = (Path("../data")).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

/tmp/ipykernel_2218630/3838749359.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.csv_loader import CSVLoader


In [2]:
# 代理设置（如不需要可自行注释掉）
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
os.environ["ALL_PROXY"] = "socks5://127.0.0.1:7890"

## 1) 准备 CSV 数据

原仓库使用的是一个 dummy 的客户信息表 `customers-100.csv`。

我们把它下载到 `../data/`，后续就可以把每一行当作一个可检索的“知识片段”。

In [3]:
def download_file(url: str, path: Path) -> Path:
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    path.write_bytes(resp.content)
    return path


csv_url = "https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/customers-100.csv"
csv_path = DATA_DIR / "customers-100.csv"

download_file(csv_url, csv_path)
csv_path

PosixPath('/data/tp/006-project/002-paper_qa/references/RAG_Techniques_CN/data/customers-100.csv')

In [4]:
# 预览 CSV（这一步不是必须，但有助于理解我们在检索什么）
df = pd.read_csv(csv_path)
df.head()

,Index,Customer Id,First Name,Last Name,Company,City,Country,Phone 1,Phone 2,Email,Subscription Date,Website
0,1,DD37Cf93aecA6Dc,Sheryl,Baxter,Rasmussen Group,East Leonard,Chile,229.077.5154,397.884.0519x718,zunigavanessa@smith.info,2020-08-24,http://www.stephenson.com/
1,2,1Ef7b82A4CAAD10,Preston,Lozano,Vega-Gentry,East Jimmychester,Djibouti,5153435776,686-620-1820x944,vmata@colon.com,2021-04-23,http://www.hobbs.com/
2,3,6F94879bDAfE5a6,Roy,Berry,Murillo-Perry,Isabelborough,Antigua and Barbuda,+1-539-402-0259,(496)978-3969x58947,beckycarr@hogan.com,2020-03-25,http://www.lawrence.com/
3,4,5Cef8BFA16c5e3c,Linda,Olsen,"Dominguez, Mcmillan and Donovan",Bensonview,Dominican Republic,001-808-617-6467x12895,+1-813-324-8756,stanleyblackwell@benson.org,2020-06-02,http://www.good-lyons.com/
4,5,053d585Ab6b3159,Joanna,Bender,"Martin, Lang and Andrade",West Priscilla,Slovakia (Slovak Republic),001-234-203-0635x76146,001-199-446-3860x3486,colinalvarado@miles.net,2021-04-17,https://goodwin-ingram.com/


## 2) 把 CSV 行变成 `Document`

这里直接用 `CSVLoader`：它会把每一行转成一个 `Document`，并把列名/列值拼成可读文本，作为 `page_content`。

对这种“每行很短”的 CSV 来说：

- 通常不需要再做 text splitting
- 每行就是一个天然的 chunk

In [5]:
loader = CSVLoader(file_path=str(csv_path))
docs = loader.load()

len(docs), docs[0]

(100,
 Document(metadata={'source': '/data/tp/006-project/002-paper_qa/references/RAG_Techniques_CN/data/customers-100.csv', 'row': 0}, page_content='Index: 1\nCustomer Id: DD37Cf93aecA6Dc\nFirst Name: Sheryl\nLast Name: Baxter\nCompany: Rasmussen Group\nCity: East Leonard\nCountry: Chile\nPhone 1: 229.077.5154\nPhone 2: 397.884.0519x718\nEmail: zunigavanessa@smith.info\nSubscription Date: 2020-08-24\nWebsite: http://www.stephenson.com/'))

## 3) 建向量库（FAISS）

把 `Document.page_content` 做 embedding 存进向量库。

后面我们会通过 `vector_store.as_retriever()` 得到一个 `retriever`，它本质上是一个 runnable：输入 query，输出一组最相关的 `Document`。

In [6]:
vector_store = FAISS.from_documents(docs, embedding=embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# quick sanity check: 看看“Sheryl Baxter”相关的行能不能被检索出来
retriever.invoke("Sheryl Baxter")[:2]

[Document(id='b7a7c6f5-db43-4221-8ac0-1ee62a7c0eaa', metadata={'source': '/data/tp/006-project/002-paper_qa/references/RAG_Techniques_CN/data/customers-100.csv', 'row': 0}, page_content='Index: 1\nCustomer Id: DD37Cf93aecA6Dc\nFirst Name: Sheryl\nLast Name: Baxter\nCompany: Rasmussen Group\nCity: East Leonard\nCountry: Chile\nPhone 1: 229.077.5154\nPhone 2: 397.884.0519x718\nEmail: zunigavanessa@smith.info\nSubscription Date: 2020-08-24\nWebsite: http://www.stephenson.com/'),
 Document(id='1ba8ea96-8eea-4196-b3d8-10b39e68a3a6', metadata={'source': '/data/tp/006-project/002-paper_qa/references/RAG_Techniques_CN/data/customers-100.csv', 'row': 8}, page_content='Index: 9\nCustomer Id: C2dE4dEEc489ae0\nFirst Name: Sheryl\nLast Name: Meyers\nCompany: Browning-Simon\nCity: Robersonstad\nCountry: Cyprus\nPhone 1: 854-138-4911x5772\nPhone 2: +1-448-910-2276x729\nEmail: mariokhan@ryan-pope.org\nSubscription Date: 2020-01-13\nWebsite: https://www.bullock.net/')]

## 4) 组合成 RAG 链：检索 → 拼 context → 生成

我们用经典的“stuff”方式：

- retriever 先找相关行
- 把这些行作为 `{context}` 塞进 prompt
- LLM 基于 context 回答

In [7]:
def format_docs(docs) -> str:
    return "\n\n".join([d.page_content for d in docs])


system_prompt = (
    "你是一个客户信息问答助手。"
    "请只使用检索到的 CSV 行（context）来回答问题。"
    "如果 context 不足以回答，就直接说不知道。"
    "回答尽量简短。\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# 兼容 langchain>=1.x：不再依赖 langchain.chains.*，用 LCEL 手动拼一个 RAG runnable。
rag_chain = (
    RunnableParallel(docs=retriever, input=RunnablePassthrough())
    | RunnablePassthrough.assign(context=lambda x: format_docs(x["docs"]))
    | RunnablePassthrough.assign(answer=prompt | llm | StrOutputParser())
)

## 5) 用自然语言查询 CSV

你可以把问题写成“查某个人/某家公司/某个国家”等自然语言形式，让 retriever 找到对应行，再让 LLM 组织成一句可读回答。

In [8]:
result = rag_chain.invoke("which company does Sheryl Baxter work for?")
result["answer"]

'Sheryl Baxter works for Rasmussen Group.'

In [9]:
# 你也可以顺便看看它检索到了哪些行（docs）
result["docs"][:2]

[Document(id='b7a7c6f5-db43-4221-8ac0-1ee62a7c0eaa', metadata={'source': '/data/tp/006-project/002-paper_qa/references/RAG_Techniques_CN/data/customers-100.csv', 'row': 0}, page_content='Index: 1\nCustomer Id: DD37Cf93aecA6Dc\nFirst Name: Sheryl\nLast Name: Baxter\nCompany: Rasmussen Group\nCity: East Leonard\nCountry: Chile\nPhone 1: 229.077.5154\nPhone 2: 397.884.0519x718\nEmail: zunigavanessa@smith.info\nSubscription Date: 2020-08-24\nWebsite: http://www.stephenson.com/'),
 Document(id='1ba8ea96-8eea-4196-b3d8-10b39e68a3a6', metadata={'source': '/data/tp/006-project/002-paper_qa/references/RAG_Techniques_CN/data/customers-100.csv', 'row': 8}, page_content='Index: 9\nCustomer Id: C2dE4dEEc489ae0\nFirst Name: Sheryl\nLast Name: Meyers\nCompany: Browning-Simon\nCity: Robersonstad\nCountry: Cyprus\nPhone 1: 854-138-4911x5772\nPhone 2: +1-448-910-2276x729\nEmail: mariokhan@ryan-pope.org\nSubscription Date: 2020-01-13\nWebsite: https://www.bullock.net/')]